# 0. Build the metadata table

Join the catalogues under `metadata/` into `metadata/params.csv`, the table every
later notebook reads, and add the galactocentric distance `ratio` of each
supernova: its projected separation from the host centre in units of the host d25
isophotal radius along the same position angle, with `ratio_err` and `R_host`.

| input | what it provides |
| --- | --- |
| `metadata/data_sn.csv` | one row per spectrum: name, coordinates, redshift, host, subtype; fixes the row order |
| `metadata/data_host.csv` | HyperLEDA properties of the host galaxies |
| `metadata/pa_ned.csv` | NED position angles for the hosts HyperLEDA leaves blank |
| `metadata/data_lc.csv` | light-curve decline rates (`Dm15`) |
| `metadata/data_spec.csv` | spectral features |

| output | |
| --- | --- |
| `metadata/params.csv` | the joined table |


## Imports

In [1]:
import numpy as np
import pandas as pd

In [2]:
import astropy.units as u
from astropy.coordinates import Distance, SkyCoord

## Configuration


In [3]:
# Input / output locations
METADATA_DIR = './metadata'

SN_PATH = f'{METADATA_DIR}/data_sn.csv'
HOST_PATH = f'{METADATA_DIR}/data_host.csv'
PA_NED_PATH = f'{METADATA_DIR}/pa_ned.csv'
LC_PATH = f'{METADATA_DIR}/data_lc.csv'
SPEC_PATH = f'{METADATA_DIR}/data_spec.csv'

PARAMS_PATH = f'{METADATA_DIR}/params.csv'

# Placeholder used throughout this project for a supernova with no measured Dm15
MISSING_DECLINE_RATE = 9.99

# HyperLEDA stores the right ascension in hours
HOURS_TO_DEGREES = 15

# `al2000` carries at most 7 decimals, so rounding the converted right ascension
# back to 9 removes the floating point noise without touching a significant digit
COORDINATE_DECIMALS = 9

In [4]:
# HyperLEDA columns that are kept, and the name they take in params.csv.
# `logdc` only feeds R_host and is dropped again by the PARAMS_COLUMNS selection.
HOST_COLUMNS = {
    'objname': 'host_name',
    'al2000': 'Host_RA',
    'de2000': 'Host_DEC',
    'logd25': 'logd25',
    'e_logd25': 'e_logd25',
    'logr25': 'logr25',
    'e_logr25': 'e_logr25',
    'pa': 'pa',
    'logdc': 'logdc',
    't': 't',
    'e_t': 't_err',
    'type': 'host_type',
    'mabs': 'host_mag',
    'modbest': 'mu',
}

LC_COLUMNS = {
    'SN_name': 'SN_name',
    'Dm15': 'D15',
    'Dm15_err': 'D15_err',
}

In [5]:
# Column order of params.csv: identity, host properties, derived distances,
# spectral subtype and features, decline rate, spectrum bookkeeping.
PARAMS_COLUMNS = [
    'SN_name', 'RA', 'DEC', 'redshift',
    'host_name', 'Host_RA', 'Host_DEC',
    'logd25', 'e_logd25', 'logr25', 'e_logr25', 'pa',
    't', 't_err', 'host_type', 'host_mag', 'mu',
    'R_host', 'ratio', 'ratio_err',
    'SN_type',
    'EW6355', 'EW6355_err', 'EW5972', 'EW5972_err',
    'min6355', 'min6355_err', 'min5972',
    'dep6355', 'dep5972', 'v_si', 'v_si_err', 'R_si',
    'D15', 'D15_err',
    'phase', 'filename_csv', 'Del_Lambda',
]

## Reading the catalogues


In [6]:
def load_supernovae(sn_path=SN_PATH):
    """Read the supernova list; its row order becomes the row order of params.csv."""
    return pd.read_csv(sn_path)

In [7]:
def load_host_properties(host_path=HOST_PATH, pa_ned_path=PA_NED_PATH):
    """Read the HyperLEDA host catalogue, keeping only the columns used here.

    HyperLEDA reports no position angle for a galaxy whose isophotes are nearly
    round, which leaves 16 of the hosts used here blank.  Those angles were
    looked up in the NED `Diameters` table instead and are supplied by
    `pa_ned_path`, which also records the catalogue each value comes from.
    """
    host = pd.read_csv(host_path)[list(HOST_COLUMNS)].rename(columns=HOST_COLUMNS)
    host['Host_RA'] = (host['Host_RA'] * HOURS_TO_DEGREES).round(COORDINATE_DECIMALS)

    pa_ned = pd.read_csv(pa_ned_path).set_index('objname')['pa']
    host['pa'] = host['pa'].fillna(host['host_name'].map(pa_ned))

    return host

In [8]:
def load_decline_rates(lc_path=LC_PATH):
    """Read the light-curve decline rates Dm15."""
    return pd.read_csv(lc_path)[list(LC_COLUMNS)].rename(columns=LC_COLUMNS)

In [9]:
def load_spectral_features(spec_path=SPEC_PATH):
    """Read the spectral features measured beforehand from the raw spectra."""
    return pd.read_csv(spec_path)

## Galactocentric distance


In [10]:
def isophotal_radius(host_logd, host_mu):
    """Physical radius of the isophote whose log10 apparent diameter is `host_logd`.

    HyperLEDA gives apparent diameters as log10 of the diameter in 0.1 arcmin,
    the convention of both `logd25` and `logdc`.  `host_mu` is the distance
    modulus of the host.
    """
    host_distance = Distance(distmod=host_mu * u.mag, unit=u.kpc)
    diameter = (10 ** (host_logd - 1)) * u.arcmin
    return host_distance * np.tan((diameter / 2).to(u.rad))

In [11]:
def diameter_fractional_error(logd25, e_logd25):
    """Turn the error on log10(d25) into a fractional error on d25 itself."""
    return (10 ** (logd25 + e_logd25) - 10 ** logd25) / 10 ** logd25

In [12]:
def distance(SN_RA, SN_DEC, host_RA, host_DEC, host_d, host_r, pa, host_mu, err):
    """Galactocentric distance of one supernova in units of the host d25 radius.

    host_d : log10 apparent d25 diameter [0.1 arcmin]
    host_r : log10 axis ratio (major axis / minor axis)
    pa     : position angle of the major axis [degree]
    host_mu: distance modulus of the host [mag]
    err    : fractional uncertainty on the apparent diameter

    Returns the ratio R_SN / R_host_at_SN and its uncertainty.
    """
    host_distance = Distance(distmod=host_mu * u.mag, unit=u.kpc)

    host_radec = SkyCoord(ra=host_RA * u.degree, dec=host_DEC * u.degree,
                          distance=host_distance, frame='icrs')
    SN_radec = SkyCoord(ra=SN_RA * u.degree, dec=SN_DEC * u.degree,
                        distance=host_distance, frame='icrs')

    # Angle between the supernova and the major axis, seen from the host centre
    SN_pa_from_galcenter = host_radec.position_angle(SN_radec).degree * u.deg
    angle_btw_SN_majoraxis = (SN_pa_from_galcenter - pa * u.deg).to(u.rad)

    # Semi-major and semi-minor axis of the d25 ellipse
    a = isophotal_radius(host_d, host_mu)
    b = a / (10 ** host_r)

    R_SN = SN_radec.separation_3d(host_radec)
    R_host_at_SN = a * b / np.sqrt(a ** 2 * np.sin(angle_btw_SN_majoraxis) ** 2
                                   + b ** 2 * np.cos(angle_btw_SN_majoraxis) ** 2)

    ratio = R_SN / R_host_at_SN
    ratio_err = R_SN / (R_host_at_SN ** 2) * R_host_at_SN * err

    return ratio, ratio_err

In [13]:
def add_galactocentric_distance(params):
    """Append `R_host`, `ratio` and `ratio_err` to the joined table."""
    err = diameter_fractional_error(params['logd25'], params['e_logd25'])

    ratios = []
    ratio_errors = []
    for i in range(len(params)):
        ratio, ratio_err = distance(params['RA'][i], params['DEC'][i],
                                    params['Host_RA'][i], params['Host_DEC'][i],
                                    params['logd25'][i], params['logr25'][i],
                                    params['pa'][i], params['mu'][i], err[i])
        ratios.append(float(ratio))
        ratio_errors.append(float(ratio_err))

    # R_host is quoted for the inclination and extinction corrected diameter
    # `logdc`, which HyperLEDA leaves blank for two hosts; those fall back on d25.
    logdc = params['logdc'].fillna(params['logd25'])

    params['R_host'] = isophotal_radius(logdc.to_numpy(), params['mu'].to_numpy()).value
    params['ratio'] = ratios
    params['ratio_err'] = ratio_errors

    return params

## Build the table


In [14]:
def build_params():
    """Join the catalogues and add the derived galactocentric distances."""
    params = (load_supernovae()
              .merge(load_host_properties(), on='host_name', how='left')
              .merge(load_decline_rates(), on='SN_name', how='left')
              .merge(load_spectral_features(), on='SN_name', how='left'))

    # A few supernovae are absent from the light-curve catalogue
    params[['D15', 'D15_err']] = params[['D15', 'D15_err']].fillna(MISSING_DECLINE_RATE)

    params = add_galactocentric_distance(params)

    return params[PARAMS_COLUMNS]

In [15]:
params = build_params()
print(f'{len(params)} spectra, {params["host_name"].nunique()} host galaxies')
params.head()

119 spectra, 118 host galaxies


,SN_name,RA,DEC,redshift,host_name,Host_RA,Host_DEC,logd25,e_logd25,logr25,...,dep6355,dep5972,v_si,v_si_err,R_si,D15,D15_err,phase,filename_csv,Del_Lambda
0,SN1994D,188.51021,7.70131,0.001494,NGC4526,188.512545,7.699261,1.842,0.018,0.449,...,0.637610,0.179055,1.056816,0.00,0.280823,1.37,0.03,0.35,SN1994D_1994-03-21_08-24-00_FLWO-1.5m_FAST_CfA...,1.467083
1,SN1994S,187.84108,29.13450,0.015177,NGC4495,187.845363,29.136442,1.128,0.035,0.373,...,0.595990,0.054295,1.021848,0.01,0.091101,0.94,0.06,1.00,SN_1994S_1994-06-16_00-00-00_Lick-3m_KAST_UCB-...,1.969612
2,SN1996ai,197.74221,37.05983,0.002900,NGC5005,197.734470,37.058994,1.683,0.025,0.500,...,0.528285,0.097199,1.046420,0.01,0.183989,0.88,0.06,-0.10,SN_1996ai_1996-06-21_00-00-00_Lick-3m_KAST_UCB...,1.992704
3,SN1996C,207.70250,49.31864,0.027000,PGC049153,207.703404,49.315106,0.899,0.064,0.257,...,0.594193,0.113131,1.029409,0.01,0.190395,0.93,0.06,1.98,1996C_1996-02-17_11-31-12_FLWO-1.5m_FAST_CfA-I...,1.430809
4,SN1996X,199.50471,-26.84592,0.008876,NGC5061,199.521255,-26.837131,1.574,0.027,0.092,...,0.661423,0.140935,1.146284,0.01,0.213078,1.26,0.04,0.31,SN1996X_1996-04-18_07-26-24_FLWO-1.5m_FAST_CfA...,1.456511


In [16]:
params.to_csv(PARAMS_PATH, index=False)
print(f'written to {PARAMS_PATH}')

written to ./metadata/params.csv
